[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msfasha/307401-Big-Data/blob/main/20252/Module%205%20-%20Introduction%20to%20Spark/7-Delta_Lake_and_the_Lakehouse.ipynb)

<h1 style="color:#E25822; font-size:2.5em;">Delta Lake &amp; the Lakehouse</h1>

**Course:** Big Data Analytics &nbsp;|&nbsp; **Introduction to Apache Spark**

---

By the end of this notebook you will be able to:

- Explain why plain Parquet falls short in production pipelines
- Describe what Delta Lake adds on top of Parquet (the `_delta_log`)
- Write and read Delta tables using the familiar PySpark API
- Use **time travel** to query any previous version of a table
- Enforce and evolve a table's schema safely
- Perform atomic **upserts** with `MERGE INTO`
- Understand how Delta Lake enables the **Lakehouse** architecture

> **Context:** Section 19 of the *Introduction to Apache Spark* document introduced Delta Lake as a concept preview. This notebook turns those concepts into hands-on practice. Full production coverage — including Databricks, Unity Catalog, and advanced optimisations — is in **Module 6**.

---

### Notebook Map

| # | Section | Key Concept |
|---|---------|-------------|
| 1 | Setup | Install `pyspark` + `delta-spark`, create SparkSession |
| 2 | The Problem with Plain Parquet | Non-atomic writes, no undo |
| 3 | What Delta Lake Adds | `_delta_log`, ACID, features overview |
| 4 | Writing & Reading Delta Tables | `format("delta")`, the transaction log |
| 5 | ACID Transactions | Atomicity, failed-write safety |
| 6 | Time Travel | `versionAsOf`, `timestampAsOf`, `DESCRIBE HISTORY` |
| 7 | Schema Enforcement | Delta rejects bad writes automatically |
| 8 | Schema Evolution | Opt-in column additions with `mergeSchema` |
| 9 | Upserts with MERGE | Update-or-insert in one atomic operation |
| 10 | Z-Ordering | Data clustering for faster queries |
| 11 | Summary & Exercises | Review + three practice problems |

---
<h2 style="color:#E25822; font-family:Arial;">1 — Environment Setup</h2>

Delta Lake requires two packages:

| Package | Purpose |
|---------|---------|
| `pyspark` | The Spark engine |
| `delta-spark` | The Delta Lake storage layer and APIs |

The `configure_spark_with_delta_pip` helper wires up the Delta extensions automatically so you do not have to pass JAR paths manually.

In [ ]:
# Install PySpark and Delta Lake (run once per Colab session)
import importlib, subprocess, sys

for pkg, import_name in [("pyspark", "pyspark"), ("delta-spark", "delta")]:
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"{pkg} installed.")
    else:
        print(f"{pkg} already available.")

In [ ]:
import os, tempfile, shutil, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# configure_spark_with_delta_pip adds the Delta Lake extensions to the builder
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("DeltaLakeIntro")
    .config("spark.sql.extensions",          "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory",           "4g")
    .config("spark.sql.shuffle.partitions",  "4")
    .config("spark.ui.showConsoleProgress",  "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Working directory for all tables in this notebook
BASE_DIR = tempfile.mkdtemp(prefix="delta_intro_")
print(f"Spark {spark.version}")
print(f"Base directory : {BASE_DIR}")
print("SparkSession ready with Delta Lake extensions.")

---
<h2 style="color:#E25822; font-family:Arial;">2 — The Problem with Plain Parquet</h2>

Parquet is a great columnar file format — fast, compressed, and widely supported. But it has no built-in concept of **transactions**. This creates three practical problems in production pipelines:

| Problem | What happens |
|---------|-------------|
| **Non-atomic overwrites** | A write that fails midway leaves partial files — readers see corrupt or incomplete data |
| **No version history** | Once you overwrite a table, the previous data is gone — no undo button |
| **No schema contract** | Any schema can be appended — silent corruption is possible |

The cell below demonstrates the first problem: a simulated partial write leaves a Parquet directory in a broken state.

In [ ]:
# ── Demonstrate the Parquet non-atomic write problem ──────────────────────
parquet_path = os.path.join(BASE_DIR, "employees_parquet")

# Version 1: write clean employee data
employees_v1 = spark.createDataFrame([
    (1, "Alice",   "Engineering", 85000.0),
    (2, "Bob",     "Marketing",   72000.0),
    (3, "Carol",   "Engineering", 91000.0),
    (4, "David",   "HR",          68000.0),
    (5, "Eve",     "Engineering", 95000.0),
], ["id", "name", "department", "salary"])

employees_v1.write.mode("overwrite").parquet(parquet_path)
print(f"V1 written: {employees_v1.count()} rows")
print(f"Files: {os.listdir(parquet_path)}")

# Simulate a partial overwrite (job crashes after writing some files)
partial_data = spark.createDataFrame([
    (6, "Frank",  "Finance",     74000.0),
    (7, "Grace",  "Marketing",   69000.0),
], ["id", "name", "department", "salary"])

# Write one partition manually to simulate crash during overwrite
try:
    # We overwrite — Spark deletes old files and starts writing new ones
    partial_data.write.mode("overwrite").parquet(parquet_path)
    print(f"\nV2 written: {partial_data.count()} rows (simulated partial overwrite)")
except Exception as e:
    print(f"Write failed: {e}")

# What a concurrent reader sees after the "crash"
df_read = spark.read.parquet(parquet_path)
print(f"\nReader sees {df_read.count()} rows after overwrite:")
df_read.show()
print("→ Original 5 rows are GONE. There is no way to get them back.")
print("→ Delta Lake solves this with atomic transactions and time travel.")

---
<h2 style="color:#E25822; font-family:Arial;">3 — What Delta Lake Adds</h2>

**Delta Lake** is an open-source storage layer that sits on top of Parquet files and adds a **transaction log** — a directory called `_delta_log/` — that records every read and write operation.

```
Without Delta Lake:          With Delta Lake:
──────────────────           ─────────────────────────────────────
 mydata/                      mydata/
   part-00000.parquet           _delta_log/           ← NEW
   part-00001.parquet             00000000000000000000.json
                                  00000000000000000001.json
                                  ...
                               part-00000.parquet
                               part-00001.parquet
```

Every write creates a new JSON entry in `_delta_log/`. Reads always go through the log to determine which files are part of the current valid snapshot.

### Features unlocked by the transaction log

| Feature | What it means in practice |
|---------|--------------------------|
| **ACID transactions** | Writes are all-or-nothing — a failed job never corrupts your table |
| **Time travel** | Query any previous version with `versionAsOf` or `timestampAsOf` |
| **Schema enforcement** | Delta rejects writes that don't match the table schema |
| **Schema evolution** | Opt-in column additions without rewriting the whole table |
| **Upserts (`MERGE`)** | Update-or-insert in one atomic operation — essential for CDC |
| **Z-ordering** | Cluster data by column to speed up filtered reads |

> **Delta Lake vs Databricks Delta:** Delta Lake is the open-source project (Apache licensed). Databricks Delta is the managed, cloud-hosted version with additional enterprise features. The PySpark API is identical.

---
<h2 style="color:#E25822; font-family:Arial;">4 — Writing &amp; Reading Delta Tables</h2>

The Delta Lake API is intentionally identical to the standard PySpark API — the only change is `format("delta")` instead of `format("parquet")`.

```python
# Parquet (before)              Delta Lake (after)
df.write                        df.write
  .format("parquet")     →        .format("delta")
  .mode("overwrite")              .mode("overwrite")
  .save(path)                     .save(path)

spark.read                      spark.read
  .format("parquet")     →        .format("delta")
  .load(path)                     .load(path)
```

This means migrating existing Parquet pipelines to Delta Lake is a one-word change.

In [ ]:
# ── Write the employee dataset as a Delta table ───────────────────────────
delta_path = os.path.join(BASE_DIR, "employees_delta")

employees = spark.createDataFrame([
    (1, "Alice",   "Engineering", 85000.0, "2021-03-15"),
    (2, "Bob",     "Marketing",   72000.0, "2020-07-01"),
    (3, "Carol",   "Engineering", 91000.0, "2019-11-20"),
    (4, "David",   "HR",          68000.0, "2022-01-10"),
    (5, "Eve",     "Engineering", 95000.0, "2018-06-05"),
], ["id", "name", "department", "salary", "hire_date"])

employees.write.format("delta").mode("overwrite").save(delta_path)

print(f"Delta table written to: {delta_path}")
print(f"\nDirectory contents:")
for entry in sorted(os.listdir(delta_path)):
    print(f"  {entry}/")
    if entry == "_delta_log":
        for log_file in sorted(os.listdir(os.path.join(delta_path, entry))):
            print(f"    └── {log_file}")

In [ ]:
# ── Read the Delta table back ─────────────────────────────────────────────
df_delta = spark.read.format("delta").load(delta_path)

print("Delta table schema:")
df_delta.printSchema()

print("Table contents:")
df_delta.show()

print(f"Row count : {df_delta.count()}")

In [ ]:
# ── Inspect the transaction log ───────────────────────────────────────────
import json as json_lib

log_dir   = os.path.join(delta_path, "_delta_log")
log_files = sorted(os.listdir(log_dir))
print(f"Transaction log files: {log_files}")

# Read the first log entry (version 0)
log_file_0 = os.path.join(log_dir, log_files[0])
with open(log_file_0) as f:
    entries = [json_lib.loads(line) for line in f if line.strip()]

print(f"\nLog entries in {log_files[0]}:")
for entry in entries:
    action = list(entry.keys())[0]
    print(f"  Action: {action}")
    if action == "metaData":
        schema = json_lib.loads(entry["metaData"]["schemaString"])
        cols   = [f["name"] for f in schema["fields"]]
        print(f"    Schema columns: {cols}")
    elif action == "add":
        print(f"    File: {entry['add']['path']}")
        print(f"    Size: {entry['add']['size']} bytes")
    elif action == "commitInfo":
        print(f"    Operation: {entry['commitInfo'].get('operation', 'N/A')}")
        print(f"    Timestamp: {entry['commitInfo'].get('timestamp', 'N/A')}")

print("\n→ Every write is recorded as a JSON entry in _delta_log/")
print("→ Delta reconstructs the current table state by replaying the log.")

---
<h2 style="color:#E25822; font-family:Arial;">5 — ACID Transactions</h2>

**ACID** stands for Atomicity, Consistency, Isolation, Durability — the four properties that define a reliable transaction.

| Property | What Delta guarantees |
|----------|-----------------------|
| **Atomicity** | A write either commits fully or not at all — no partial writes |
| **Consistency** | The table always moves from one valid state to another |
| **Isolation** | Concurrent readers always see a consistent snapshot |
| **Durability** | Once committed, data survives failures |

In practice, **Atomicity** is the one you feel most directly: if a Spark job writing to a Delta table fails halfway through, the table is unchanged. The failed write simply doesn't appear in the transaction log.

```
Job starts writing →  new Parquet files land in the directory
                   →  if job crashes: log entry is NEVER written
                   →  readers ignore uncommitted files (log is the source of truth)
                   →  if job succeeds: log entry is written atomically
                   →  all readers immediately see the new version
```

In [ ]:
# ── Demonstrate ACID: multiple overwrites, table always consistent ─────────
from delta.tables import DeltaTable

acid_path = os.path.join(BASE_DIR, "acid_demo")

# Version 0 — initial write
batch_0 = spark.createDataFrame([
    (101, "Widget A", 29.99),
    (102, "Widget B", 49.99),
    (103, "Widget C", 19.99),
], ["product_id", "name", "price"])

batch_0.write.format("delta").mode("overwrite").save(acid_path)
print("Version 0 written.")

# Version 1 — a price update (full overwrite)
batch_1 = spark.createDataFrame([
    (101, "Widget A", 34.99),   # price increased
    (102, "Widget B", 49.99),
    (103, "Widget C", 19.99),
    (104, "Widget D", 59.99),   # new product
], ["product_id", "name", "price"])

batch_1.write.format("delta").mode("overwrite").save(acid_path)
print("Version 1 written.")

# A reader always sees only committed versions
df_current = spark.read.format("delta").load(acid_path)
print(f"\nCurrent table (version 1, {df_current.count()} rows):")
df_current.show()

# Simulate a FAILED write: we catch the exception before committing
print("Simulating a failed write (exception before commit)...")
try:
    df_bad = spark.createDataFrame([(999, "Broken", -1.0)], ["product_id", "name", "price"])
    df_bad.write.format("delta").mode("overwrite").save(acid_path)
    raise RuntimeError("Simulated job failure!")
except RuntimeError as e:
    print(f"  Job failed: {e}")

# Show the table is unchanged
df_after_fail = spark.read.format("delta").load(acid_path)
print(f"\nTable after failed write ({df_after_fail.count()} rows — unchanged):")
df_after_fail.show()

---
<h2 style="color:#E25822; font-family:Arial;">6 — Time Travel</h2>

Because every version of a Delta table is recorded in the transaction log, you can **query any previous version** — without maintaining separate backup copies.

```python
# By version number
spark.read.format("delta")
     .option("versionAsOf", 0)
     .load(path)

# By timestamp
spark.read.format("delta")
     .option("timestampAsOf", "2025-01-01 00:00:00")
     .load(path)

# Full history
from delta.tables import DeltaTable
DeltaTable.forPath(spark, path).history().show()
```

### When time travel is useful

| Use case | How |
|----------|-----|
| Audit / compliance | Reproduce the exact table state at any point in time |
| Rollback a bad write | Read the last good version and write it back |
| Reproducible ML | Pin training data to a specific version to guarantee reproducibility |
| Debugging | Compare current state to a previous version to spot unexpected changes |

In [ ]:
# ── Build up multiple versions of the employees table ─────────────────────
tt_path = os.path.join(BASE_DIR, "employees_tt")

# Version 0: original 5 employees
v0 = spark.createDataFrame([
    (1, "Alice",   "Engineering", 85000.0),
    (2, "Bob",     "Marketing",   72000.0),
    (3, "Carol",   "Engineering", 91000.0),
    (4, "David",   "HR",          68000.0),
    (5, "Eve",     "Engineering", 95000.0),
], ["id", "name", "department", "salary"])

v0.write.format("delta").mode("overwrite").save(tt_path)
ts_v0 = time.time()
print(f"Version 0 written: {v0.count()} rows")

time.sleep(1)   # ensure distinct timestamps

# Version 1: add two new hires
v1 = spark.createDataFrame([
    (1, "Alice",   "Engineering", 85000.0),
    (2, "Bob",     "Marketing",   72000.0),
    (3, "Carol",   "Engineering", 91000.0),
    (4, "David",   "HR",          68000.0),
    (5, "Eve",     "Engineering", 95000.0),
    (6, "Frank",  "Finance",      74000.0),   # new hire
    (7, "Grace",  "Marketing",    69000.0),   # new hire
], ["id", "name", "department", "salary"])

v1.write.format("delta").mode("overwrite").save(tt_path)
print(f"Version 1 written: {v1.count()} rows  (+2 new hires)")

time.sleep(1)

# Version 2: give Engineering a raise
v2 = v1.withColumn(
    "salary",
    F.when(F.col("department") == "Engineering", F.col("salary") * 1.10)
     .otherwise(F.col("salary"))
)
v2.write.format("delta").mode("overwrite").save(tt_path)
print(f"Version 2 written: {v2.count()} rows  (Engineering +10% salary)")

In [ ]:
# ── Query by version number ────────────────────────────────────────────────
for version in [0, 1, 2]:
    df_v = spark.read.format("delta").option("versionAsOf", version).load(tt_path)
    avg_sal = df_v.agg(F.avg("salary").alias("avg_sal")).collect()[0]["avg_sal"]
    print(f"Version {version}: {df_v.count()} rows  |  avg salary = ${avg_sal:,.0f}")

print()

# Compare Engineering salaries across versions
print("Engineering salaries across versions:")
print(f"  {'Name':<8} {'V0 salary':>12} {'V2 salary':>12} {'Change':>10}")
print("  " + "-" * 46)

df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(tt_path)
df_v2 = spark.read.format("delta").option("versionAsOf", 2).load(tt_path)

eng_v0 = {row["name"]: row["salary"]
          for row in df_v0.filter(F.col("department") == "Engineering").collect()}
eng_v2 = {row["name"]: row["salary"]
          for row in df_v2.filter(F.col("department") == "Engineering").collect()}

for name in sorted(eng_v0):
    s0, s2 = eng_v0[name], eng_v2.get(name, 0)
    print(f"  {name:<8} ${s0:>11,.0f} ${s2:>11,.0f}  +{(s2-s0):>7,.0f}")

In [ ]:
# ── Inspect the full table history ───────────────────────────────────────
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, tt_path)
print("Full table history:")
dt.history().select("version", "timestamp", "operation", "operationParameters").show(
    truncate=False
)

# Rollback: restore version 0 by writing it back
print("\nRolling back to version 0...")
df_rollback = spark.read.format("delta").option("versionAsOf", 0).load(tt_path)
df_rollback.write.format("delta").mode("overwrite").save(tt_path)

# Confirm rollback
df_current = spark.read.format("delta").load(tt_path)
print(f"After rollback: {df_current.count()} rows (original 5)")
df_current.show()

---
<h2 style="color:#E25822; font-family:Arial;">7 — Schema Enforcement</h2>

Delta Lake records the table's schema in the `_delta_log` and **rejects any write** whose schema is incompatible — preventing silent data corruption.

```
Existing table columns:  id, name, department, salary
Write attempt columns:   id, name, department, salary, unexpected_bonus  ← REJECTED
```

The rejection happens automatically without any configuration — it is on by default.

This is especially important in pipelines where:
- Multiple teams write to the same table
- An upstream schema change is not communicated
- A bug adds or renames a column unexpectedly

In [ ]:
# ── Schema enforcement: Delta rejects incompatible writes ─────────────────
se_path = os.path.join(BASE_DIR, "schema_enforcement")

# Write the baseline table
base = spark.createDataFrame([
    (1, "Alice", "Engineering", 85000.0),
    (2, "Bob",   "Marketing",   72000.0),
], ["id", "name", "department", "salary"])

base.write.format("delta").mode("overwrite").save(se_path)
print(f"Baseline table written: {base.dtypes}")

# Attempt 1: write with an EXTRA column → should be rejected
print("\nAttempt 1: write with extra column 'bonus' ...")
df_extra_col = spark.createDataFrame([
    (3, "Carol", "Engineering", 91000.0, 5000.0),
], ["id", "name", "department", "salary", "bonus"])

try:
    df_extra_col.write.format("delta").mode("append").save(se_path)
    print("  Write succeeded (unexpected)")
except Exception as e:
    print(f"  REJECTED: {type(e).__name__}")
    print(f"  Message : {str(e)[:120]}...")

# Attempt 2: write with WRONG type for 'id' (string instead of long) → rejected
print("\nAttempt 2: write with wrong type (id as string instead of long) ...")
df_wrong_type = spark.createDataFrame([
    ("THREE", "Carol", "Engineering", 91000.0),
], ["id", "name", "department", "salary"])

try:
    df_wrong_type.write.format("delta").mode("append").save(se_path)
    print("  Write succeeded (unexpected)")
except Exception as e:
    print(f"  REJECTED: {type(e).__name__}")
    print(f"  Message : {str(e)[:120]}...")

# Confirm table is still clean
df_clean = spark.read.format("delta").load(se_path)
print(f"\nTable after rejected writes: {df_clean.count()} rows (unchanged)")
df_clean.show()

---
<h2 style="color:#E25822; font-family:Arial;">8 — Schema Evolution</h2>

Schema enforcement is the default — but sometimes you genuinely **need** to add a new column to an existing table. Delta Lake supports opt-in schema evolution with the `mergeSchema` option.

```python
# This would be REJECTED by default (extra column)
new_data.write.format("delta").mode("append").save(path)

# This is ACCEPTED — Delta adds the new column to the schema
new_data.write.format("delta").mode("append") \
    .option("mergeSchema", "true") \
    .save(path)
```

Rows that existed before the new column was added will show `null` for that column — no rewrite of historical data is needed.

### What schema evolution supports

| Change | Supported |
|--------|:---------:|
| Add new column | ✓ |
| Rename existing column | ✗ (treated as drop + add) |
| Change column data type | ✗ (requires explicit overwrite) |
| Drop a column | ✗ (requires `overwriteSchema=true`) |

In [ ]:
# ── Schema evolution with mergeSchema ─────────────────────────────────────
evo_path = os.path.join(BASE_DIR, "schema_evolution")

# Version 0: original schema
original = spark.createDataFrame([
    (1, "Alice", "Engineering", 85000.0),
    (2, "Bob",   "Marketing",   72000.0),
], ["id", "name", "department", "salary"])

original.write.format("delta").mode("overwrite").save(evo_path)
print("Original schema:")
spark.read.format("delta").load(evo_path).printSchema()

# Append data with a new column 'years_exp' using mergeSchema
evolved = spark.createDataFrame([
    (3, "Carol", "Engineering", 91000.0, 7),
    (4, "David", "HR",          68000.0, 3),
], ["id", "name", "department", "salary", "years_exp"])

evolved.write.format("delta").mode("append")     .option("mergeSchema", "true")     .save(evo_path)

print("Schema after evolution (mergeSchema=true):")
spark.read.format("delta").load(evo_path).printSchema()

# Show the full table — rows without years_exp get null
print("Full table after schema evolution:")
spark.read.format("delta").load(evo_path).show()
print("→ Rows 1 and 2 show null for years_exp (existed before the column was added)")

---
<h2 style="color:#E25822; font-family:Arial;">9 — Upserts with MERGE</h2>

**Upsert** means "update if the row exists, insert if it doesn't." This is one of the most common operations in data engineering — used whenever incoming data contains both new records and corrections to existing ones (Change Data Capture / CDC pattern).

Plain Parquet has no efficient upsert support: you would have to read the entire table, merge in memory, then rewrite. For large tables this is prohibitively slow.

Delta Lake's `MERGE INTO` solves this in one atomic operation:

```python
from delta.tables import DeltaTable

target = DeltaTable.forPath(spark, path)

target.alias("t").merge(
    source.alias("s"),
    "t.id = s.id"                    # match condition
).whenMatchedUpdateAll()             # if match → update all columns
 .whenNotMatchedInsertAll()          # if no match → insert new row
 .execute()
```

The operation is **atomic**: readers see either the old state or the fully merged new state — never a halfway result.

In [ ]:
# ── MERGE (Upsert) demo ────────────────────────────────────────────────────
from delta.tables import DeltaTable

merge_path = os.path.join(BASE_DIR, "employees_merge")

# ── Initial target table ───────────────────────────────────────────────────
target_data = spark.createDataFrame([
    (1, "Alice",  "Engineering", 85000.0, "active"),
    (2, "Bob",    "Marketing",   72000.0, "active"),
    (3, "Carol",  "Engineering", 91000.0, "active"),
    (4, "David",  "HR",          68000.0, "active"),
    (5, "Eve",    "Engineering", 95000.0, "active"),
], ["id", "name", "department", "salary", "status"])

target_data.write.format("delta").mode("overwrite").save(merge_path)
print("Target table (before MERGE):")
spark.read.format("delta").load(merge_path).show()

# ── Incoming source: HR system changes ────────────────────────────────────
# Rows 2 and 4 have changes; rows 6 and 7 are new hires
source_changes = spark.createDataFrame([
    (2, "Bob",    "Marketing",   78000.0, "active"),   # salary increase
    (4, "David",  "Finance",     71000.0, "active"),   # department change
    (6, "Frank",  "Finance",     74000.0, "active"),   # new hire
    (7, "Grace",  "Marketing",   69000.0, "active"),   # new hire
], ["id", "name", "department", "salary", "status"])

print("Incoming changes (source):")
source_changes.show()

# ── Apply MERGE ─────────────────────────────────────────────────────────────
target_dt = DeltaTable.forPath(spark, merge_path)

(
    target_dt.alias("target")
    .merge(source_changes.alias("source"), "target.id = source.id")
    .whenMatchedUpdateAll()      # update existing rows
    .whenNotMatchedInsertAll()   # insert new rows
    .execute()
)

print("Target table (after MERGE):")
spark.read.format("delta").load(merge_path).show()
print("→ Bob's salary updated, David's department changed, Frank and Grace inserted.")

In [ ]:
# ── MERGE with conditional logic (soft deletes) ───────────────────────────
# Real pipelines often need more nuance:
#   - Update salary only if the new value is higher
#   - Mark departures as inactive instead of deleting

departures = spark.createDataFrame([
    (3, "Carol", "Engineering", 91000.0, "resigned"),  # Carol is leaving
], ["id", "name", "department", "salary", "status"])

target_dt2 = DeltaTable.forPath(spark, merge_path)

(
    target_dt2.alias("t")
    .merge(departures.alias("s"), "t.id = s.id")
    # Conditional update: only change status column
    .whenMatchedUpdate(set={"status": "s.status"})
    .execute()
)

print("After marking Carol as resigned:")
spark.read.format("delta").load(merge_path).show()
print("→ MERGE gives fine-grained control: update only specific columns,")
print("   apply conditions, or handle deletes as status flags (soft delete).")

---
<h2 style="color:#E25822; font-family:Arial;">10 — Z-Ordering</h2>

**Z-ordering** is a data-skipping technique that physically co-locates related rows in the same Parquet files. When you query with a filter on a Z-ordered column, Delta Lake can skip entire files that cannot contain matching rows — dramatically reducing the amount of data read.

```python
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, path)
dt.optimize().executeZOrderBy("department")
```

```
Without Z-order:             With Z-order (by department):
──────────────────           ──────────────────────────────
 file_0: Eng, HR, Mkt        file_0: Engineering rows only
 file_1: Fin, Eng, HR        file_1: HR + Finance rows only
 file_2: Mkt, Fin, Eng       file_2: Marketing rows only
                             
 Query: WHERE department      Query: WHERE department
        = 'Engineering'              = 'Engineering'
 → must read 3 files          → reads only file_0  ✓
```

### When to use Z-ordering

| Situation | Benefit |
|-----------|---------|
| High-cardinality filter columns (city, category, user_id) | Skip irrelevant files |
| Columns frequently used in WHERE clauses | Reduce scan size significantly |
| Large tables (>1 GB) | Overhead worthwhile at scale |

> **Note:** Z-ordering requires running `OPTIMIZE`, which rewrites Parquet files. On small tables (like the ones in this notebook) the overhead isn't worth it — but the API is shown below so the syntax is familiar.

In [ ]:
# ── Z-Ordering example ────────────────────────────────────────────────────
zo_path = os.path.join(BASE_DIR, "employees_zorder")

# Write a larger table to make Z-ordering meaningful
departments = ["Engineering", "Marketing", "Finance", "HR", "Operations"]
large_df = (
    spark.range(5000)
    .withColumn("name",       F.concat(F.lit("emp_"), F.col("id").cast("string")))
    .withColumn("department", F.element_at(
        F.array(*[F.lit(d) for d in departments]),
        (F.col("id") % 5 + 1).cast("int")))
    .withColumn("salary",     (F.rand(seed=42) * 60000 + 40000).cast("double"))
    .withColumn("status",     F.lit("active"))
    .select("id", "name", "department", "salary", "status")
)

large_df.write.format("delta").mode("overwrite").save(zo_path)

files_before = len([f for f in os.listdir(zo_path) if f.endswith(".parquet")])
print(f"Files before OPTIMIZE: {files_before}")

# Run OPTIMIZE with Z-order on 'department'
dt_zo = DeltaTable.forPath(spark, zo_path)
dt_zo.optimize().executeZOrderBy("department")

files_after = len([f for f in os.listdir(zo_path) if f.endswith(".parquet")])
print(f"Files after  OPTIMIZE: {files_after}")
print("(OPTIMIZE compacts small files AND applies Z-order co-location)")

# Query with a filter — Delta uses file statistics to skip irrelevant files
t0 = time.time()
result = spark.read.format("delta").load(zo_path)     .filter(F.col("department") == "Engineering")     .agg(F.avg("salary").alias("avg_salary"))     .collect()
print(f"\nEngineering avg salary: ${result[0]['avg_salary']:,.0f}")
print(f"Query time: {time.time()-t0:.2f}s")
print("\n→ On large production tables Z-ordering reduces query time by 2–10×")
print("   for selective filters on the Z-ordered column.")

---
<h2 style="color:#E25822; font-family:Arial;">11 — Delta Lake vs Plain Parquet — Summary</h2>

| Feature | Plain Parquet | Delta Lake |
|---------|:------------:|:----------:|
| **Columnar storage** | ✓ | ✓ (same Parquet files underneath) |
| **Compression** | ✓ | ✓ |
| **ACID transactions** | ✗ | ✓ |
| **Atomic writes** | ✗ | ✓ |
| **Time travel** | ✗ | ✓ |
| **Schema enforcement** | ✗ | ✓ (by default) |
| **Schema evolution** | ✗ | ✓ (opt-in) |
| **Upserts (MERGE)** | ✗ | ✓ |
| **Z-ordering / data skipping** | ✗ | ✓ |
| **Audit history** | ✗ | ✓ |
| **Compatible with existing Spark code** | ✓ | ✓ (format change only) |

### The Lakehouse Pattern

Delta Lake is the storage foundation of the **Lakehouse** architecture — a design that combines the scalability and low cost of a data lake with the reliability and BI capabilities of a data warehouse.

```
Traditional stack:                Lakehouse with Delta Lake:
──────────────────                ──────────────────────────────────────────
  Raw data lake                     ONE storage layer
  (S3 / ADLS / GCS)                 Parquet files + _delta_log/
       +                   ──►      
  Data warehouse                    ACID ✓  BI queries ✓  ML training ✓
  (Redshift / Synapse)              Streaming ✓  Schema enforcement ✓
  (expensive, duplicate)            (no expensive duplication)
```

**In industry:** Databricks (the company behind Delta Lake), Azure Synapse, and AWS Lake Formation all use Delta tables as their primary storage layer. When you join a data engineering team, "Delta table" will be as common a term as "DataFrame".

---
<h2 style="color:#E25822; font-family:Arial;">Exercises</h2>

Work through each exercise independently. Use the concepts and code patterns from this notebook as a reference.

---

### Exercise 1 — Time Travel & Rollback (Beginner)

1. Create a Delta table called `products` at a path of your choice with columns: `product_id` (int), `name` (string), `category` (string), `price` (double). Populate it with at least 5 rows.
2. Write a **second version**: increase all prices by 15%.
3. Write a **third version**: add 3 new products.
4. Use `DeltaTable.forPath(spark, path).history()` to display the full version history.
5. Query **version 0** (original prices) and calculate the average price per category.
6. **Rollback**: restore the table to version 1 (after the 15% increase, but before the new products). Confirm the row count is back to 5.

---

### Exercise 2 — Schema Enforcement & Evolution (Intermediate)

1. Create a Delta table `inventory` with columns: `item_id` (int), `item_name` (string), `quantity` (int), `warehouse` (string).
2. **Schema enforcement:** Attempt to append a DataFrame that includes an extra column `supplier_code` (string). Catch the exception and print a clear message explaining why it was rejected.
3. **Schema evolution:** Use `mergeSchema=true` to successfully append the same data (with `supplier_code`). Confirm the new schema with `printSchema()`.
4. Show the full table. Verify that original rows have `null` for `supplier_code` and new rows have the actual values.
5. Inspect the transaction log history — how many versions exist, and what operation created the schema change?

---

### Exercise 3 — MERGE / CDC Pipeline (Advanced)

Simulate a CDC (Change Data Capture) pipeline for a `customers` table.

1. Create the **target** Delta table `customers` with columns: `customer_id`, `name`, `email`, `tier` (Bronze/Silver/Gold), `last_updated` (string date). Populate with 6 customers.
2. Create a **source** DataFrame representing today's CDC feed:
   - 2 existing customers with updated email addresses
   - 1 existing customer upgraded to Gold tier
   - 3 brand-new customers
3. Apply `MERGE INTO` so that:
   - Existing customers are updated (all columns)
   - New customers are inserted
4. Show the merged table and verify: 9 total customers, correct emails and tiers.
5. **Bonus:** Add a fourth MERGE clause `.whenNotMatchedBySource()` that marks any customer **not** in today's CDC feed as `tier = 'Inactive'`. *(Hint: this requires Delta Lake 2.3+ — check if it works in your environment.)*

In [ ]:
# ── Exercise 1 scaffold ────────────────────────────────────────────────────
ex1_path = os.path.join(BASE_DIR, "ex1_products")

# 1. Create initial products table
# TODO: create a DataFrame with at least 5 products and write as Delta
# products_v0 = spark.createDataFrame([...], ["product_id", "name", "category", "price"])
# products_v0.write.format("delta").mode("overwrite").save(ex1_path)

# 2. Version 2: increase all prices by 15%
# TODO: read the Delta table, apply price * 1.15, write back as overwrite

# 3. Version 3: add 3 new products
# TODO: append 3 new rows using mode("append")

# 4. Show full version history
# TODO: DeltaTable.forPath(spark, ex1_path).history().show()

# 5. Query version 0 — average price per category
# TODO: read versionAsOf=0, groupBy category, agg avg price

# 6. Rollback to version 1
# TODO: read versionAsOf=1, write back as overwrite, confirm row count

print("Exercise 1 scaffold loaded — implement the TODOs above.")

In [ ]:
# ── Exercise 2 scaffold ────────────────────────────────────────────────────
ex2_path = os.path.join(BASE_DIR, "ex2_inventory")

# 1. Create inventory table
# TODO: create DataFrame with item_id, item_name, quantity, warehouse
# inventory = spark.createDataFrame([...], ["item_id", "item_name", "quantity", "warehouse"])
# inventory.write.format("delta").mode("overwrite").save(ex2_path)

# 2. Attempt append with extra column (should fail)
# TODO: create df with extra 'supplier_code' column, try append, catch exception

# 3. Use mergeSchema to evolve the schema
# TODO: write same df with .option("mergeSchema", "true")

# 4. Show full table and verify nulls for original rows
# TODO: spark.read.format("delta").load(ex2_path).show()

# 5. Show history
# TODO: DeltaTable.forPath(spark, ex2_path).history().show()

print("Exercise 2 scaffold loaded — implement the TODOs above.")

In [ ]:
# ── Exercise 3 scaffold ────────────────────────────────────────────────────
ex3_path = os.path.join(BASE_DIR, "ex3_customers")

# 1. Create initial customers table (6 rows)
# TODO: create customers DataFrame with customer_id, name, email, tier, last_updated
# customers = spark.createDataFrame([...], [...])
# customers.write.format("delta").mode("overwrite").save(ex3_path)

# 2. Create CDC source feed
# TODO: 2 email updates + 1 tier upgrade + 3 new customers
# cdc_feed = spark.createDataFrame([...], [...])

# 3. Apply MERGE
# TODO:
# target = DeltaTable.forPath(spark, ex3_path)
# ( target.alias("t")
#   .merge(cdc_feed.alias("s"), "t.customer_id = s.customer_id")
#   .whenMatchedUpdateAll()
#   .whenNotMatchedInsertAll()
#   .execute() )

# 4. Show merged table and verify 9 rows
# TODO: spark.read.format("delta").load(ex3_path).show()

# 5. Bonus: whenNotMatchedBySource soft-delete
# TODO (optional, Delta 2.3+)

print("Exercise 3 scaffold loaded — implement the TODOs above.")

In [ ]:
# ── Cleanup: stop SparkSession and remove temp files ──────────────────────
spark.stop()
shutil.rmtree(BASE_DIR, ignore_errors=True)
print(f"SparkSession stopped.")
print(f"Temporary directory removed: {BASE_DIR}")
print("Notebook complete.")